In [1]:
import pandas as pd
import re
import requests
import time
import json
import os

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

def query_ensembl_grch37(chrom, pos, max_retries=3):
    url = f"https://grch37.rest.ensembl.org/overlap/region/human/{chrom}:{int(pos)-1}-{int(pos)+1}?feature=gene;content-type=application/json"
    headers = {"User-Agent": "Mozilla/5.0 (research script)"}
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            if resp.status_code == 200:
                return resp.json()
            time.sleep(2 * (attempt + 1))
        except Exception:
            time.sleep(2 * (attempt + 1))
    return None

# --- AA Stage 3 pre-PC shortlist: already fully annotated (9 SNPs) ---
aa_stage3_genes = {"FAM126A", "ZNF79", "CASC8", "LAMA3", "PPP1R12B", "FLG-AS1", "CELA1", "PRSS54", "TAS1R3"}
print("AA Stage 3 (pre-PC) genes:", aa_stage3_genes)

# --- EA Stage 3 pre-PC shortlist: annotate all 18, not just the 5 PC survivors ---
out_dir = r"C:\Users\user\Downloads\GSE148812_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
pos_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]

ea_shortlist = pd.read_csv(os.path.join(out_dir, "shortlist_smoking_100pct_final.csv"))
print("EA Stage 3 (pre-PC) SNP count:", len(ea_shortlist))

ea_gene_map = {}
for snp in ea_shortlist["probe_id"]:
    core = strip_address_suffix(snp)
    if core not in pos_lookup.index:
        ea_gene_map[snp] = "UNKNOWN"
        continue
    chrom, pos = pos_lookup.loc[core, "Chr"], pos_lookup.loc[core, "MapInfo"]
    data = query_ensembl_grch37(chrom, pos)
    ea_gene_map[snp] = data[0].get("external_name", "UNKNOWN") if data else f"intergenic_chr{chrom}"
    time.sleep(0.3)

for snp, gene in ea_gene_map.items():
    print(f"{snp[:35]:37s} -> {gene}")

ea_stage3_genes = set(ea_gene_map.values())

# --- Compare ---
print("\n=== TEST 1 RESULT ===")
print("AA Stage 3 genes:", aa_stage3_genes)
print("EA Stage 3 genes:", ea_stage3_genes)
overlap = aa_stage3_genes & ea_stage3_genes
print("Overlap (pre-PC):", overlap)
print("Case A (overlap exists pre-PC)" if overlap else "Case B (no overlap even pre-PC)")

with open(os.path.join(out_dir, "test1_gene_overlap_result.json"), "w") as f:
    json.dump({"AA": list(aa_stage3_genes), "EA": list(ea_stage3_genes), "overlap": list(overlap)}, f)

AA Stage 3 (pre-PC) genes: {'LAMA3', 'CASC8', 'CELA1', 'TAS1R3', 'PRSS54', 'PPP1R12B', 'ZNF79', 'FAM126A', 'FLG-AS1'}
EA Stage 3 (pre-PC) SNP count: 18
exm71047-0_B_R_1921357564             -> SPATA1
exm112878-0_B_F_1921488545            -> OR10X1
exm935491-0_T_R_1918372056            -> CPT1A
exm2251364-0_T_F_1975257170           -> SACS
exm1093486-0_B_R_1922791026           -> NOP9
exm1242904-0_T_F_1921733541           -> CETP
exm1286060-0_B_F_1918731497           -> DVL2
exm1387779-0_B_F_1921595005           -> POLI
exm1395687-0_T_R_1923304216           -> THEG
exm1413456-0_B_F_1923224907           -> MLLT1
exm1441545-0_B_R_2058868341           -> AC010646.3
exm183025-0_T_F_1918992921            -> TRMT61B
exm421337-0_B_F_1923050859            -> USP53
exm579132-0_T_R_1921991094            -> TAAR6
exm619924-0_B_F_1918533121            -> HUS1
exm634138-0_B_F_1918555129            -> PEX1
exm695084-0_B_R_1922987914            -> UNC5D
exm793377-0_T_R_1922407882            -> ADAMTS1

In [2]:
# --- AA CPD Stage 3 pre-PC shortlist ---
aa_cpd_stage3_genes = {"PPP1R12B", "HEMK1", None}  # third SNP was intergenic (chr4), no gene
aa_cpd_stage3_genes.discard(None)
print("AA CPD Stage 3 (pre-PC) genes:", aa_cpd_stage3_genes)

# --- EA CPD Stage 3 pre-PC shortlist: annotate all 4 (already have from earlier work) ---
ea_cpd_stage3_genes = {"AHRR", "EMILIN2", "PSG10P", "ARFGAP3"}
print("EA CPD Stage 3 (pre-PC) genes:", ea_cpd_stage3_genes)

overlap_cpd = aa_cpd_stage3_genes & ea_cpd_stage3_genes
print("\n=== CPD TEST 1 RESULT ===")
print("Overlap (pre-PC):", overlap_cpd)
print("Case A (overlap exists pre-PC)" if overlap_cpd else "Case B (no overlap even pre-PC)")

AA CPD Stage 3 (pre-PC) genes: {'PPP1R12B', 'HEMK1'}
EA CPD Stage 3 (pre-PC) genes: {'EMILIN2', 'AHRR', 'ARFGAP3', 'PSG10P'}

=== CPD TEST 1 RESULT ===
Overlap (pre-PC): set()
Case B (no overlap even pre-PC)
